In [ ]:
import pymc as pm
import arviz as az
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal
import pytensor
import pytensor.tensor as pt

np.random.seed(42)
T = 100
rainfall = np.random.uniform(0, 20, T)
runoff_coeff = 0.5

noise = np.random.normal(0, 1, T)
exog = runoff_coeff * rainfall + noise

exog[0] = 0
flow_c = scipy.signal.lfilter(b=[1.0], a=[1.0, -0.8], x=exog)
sensor_noise = np.random.normal(0, 2, T) # The sensor is shaking
flow_true = flow_c + sensor_noise

flow_centered = flow_true - flow_true.mean()
rainfall_centered = rainfall-rainfall.mean()



In [ ]:
with pm.Model() as m3_arx:
    rain_data = pm.Data("rain_data", rainfall_centered)
    flow_data = pm.Data("flow_data", flow_centered)


    phi = pm.Uniform("phi", 0.0, 1.0)
    #0.5+-0.5 i.e model could be between amnesiac and grw but not <0 or >1

    beta = pm.HalfNormal("beta", sigma=1)
    # beta here is the runoff coeff, we know its positive, we could model as pm.uniform (0,1 bounds)

    #exg = beta * rain_data
    # external exogenous

    sigma_process = pm.HalfNormal("sigma_process", sigma=1)
    # because we are operating on fluctuating component
    sigma_obs = pm.HalfNormal("sigma_obs", sigma=1)
    # because our observed is also centered data not full data

    # NCP Architecture: Separate the magnitude from the distribution
    # z score can come from student, fixed nu =4, to avoid non-identifiability loop due to flexible nu on both sensor (likelihood)
    # and on the actual phyiscs, 4 for 'robust modeling'
    # why fix the river? cuz river despite being chaotic follows laws of nature
    # sensor can show 0 -1000000 or anything
    # DO NOT HARDCODE SHAPE=T for out-of-sample predictions. Let it infer from the data length!

    raw_shocks = pm.StudentT("raw_shocks", mu=0, sigma=1, nu=4, shape=rain_data.shape[0])
    # equiv to len(rain_data) but the var is pytensor var so use shape, not py var

    process_shocks = pm.Deterministic("shocks", raw_shocks * sigma_process)

    initial_flow = pm.Normal("initial_flow", mu=0, sigma=sigma_process)

    def ar_ex_physics(today_rain, today_shock, yesterday_flow, phi, beta):
        today_flow = (phi * yesterday_flow) + (beta * today_rain) + today_shock
        return today_flow


    latent_flow, _ = pytensor.scan( # Using pytensor.scan directly
        fn=ar_ex_physics,
        sequences=[rain_data, process_shocks],
        outputs_info=[initial_flow],
        non_sequences=[phi, beta] # Fixed typo from non_sequenuces
    )
    #baseline day 0

    # positional arguements

    latent_flow = pm.Deterministic("latent_flow", latent_flow)


    # Student-T with nu <= 2 has INFINITE variance.
    # We use a shifted Gamma prior to act as a mathematical firewall, guaranteeing
    # that while the model anticipates extreme shocks (fat tails), the variance
    # of the Himalayan hydrologic system remains structurally finite and calculable.
    nu = pm.Gamma("nu", alpha=2, beta=0.1) + 2.0

    likelihood = pm.StudentT("likelihood", mu = latent_flow  , sigma = sigma_obs, nu =nu, observed=flow_data)
    # not river data but the errors in our yt that model compares against observed

    # if we define mu = ar+exg instead of using const=exg in pm.AR
    # it means that we are defining the exg NOT ncesessarily as sth that would physically appear/reality
    # but as 'modification' to the sensor data

    #  this is seperation of Latent State (the real physics) from the Observation Equation (the likelihood/sensor)

    prior_checks = pm.sample_prior_predictive(samples=1000, random_seed=42)

display(pm.model_to_graphviz(m3_arx))
az.plot_ppc(prior_checks, group="prior", kind="kde", observed=True)



# good priors, because its a centered data, clustered around 0 and okay to bleed to negative symmetrical

In [ ]:
with m3_arx:
    trace = pm.sample(draws=2000,
                          tune=1000,
                          cores=2, chains=2, target_accept=0.95,
                          random_seed=42,
                          progressbar=False,)

    pm.sample_posterior_predictive(trace, random_seed=42, progressbar=False, extend_inferencedata=True)
    pm.compute_log_likelihood(trace)

display(az.summary(trace, var_names=["phi", "beta", "initial_flow", "sigma_obs", "sigma_process"]))
az.plot_trace(trace, var_names=["phi", "beta", "initial_flow", "sigma_obs", "sigma_process"], compact=False)

#az.plot_energy(trace)
#az.plot_pair(trace, divergences=True)
# no need to look energy and pair if divergence =1

# Put these on their own figures so they don't crash into each other!

az.plot_khat(az.loo(trace), show_bins=True, show_hlines=True)

az.plot_ppc(trace, kind='cumulative', observed=True)
plt.show()

pred_mean = trace.posterior_predictive["likelihood"].mean(dim=["chain", "draw"])
residuals = flow_centered - pred_mean.values
plt.acorr(residuals, maxlags=20)
plt.title("Residual Autocorrelation")
plt.show()

In [ ]:
# timeline plot

fig, ax = plt.subplots(figsize=(12, 5))

# Plot the ACTUAL observations, not the centered ones
ax.plot(flow_true, 'k.', label="Sensor Data (Observed)", alpha=0.5)

# The missing baseline
base_level = flow_true.mean()

# Get the Latent Flow (The Hidden River Physics) AND UN-CENTER IT
latent_hdi = az.hdi(trace.posterior["latent_flow"]).latent_flow + base_level
latent_mean = trace.posterior["latent_flow"].mean(dim=["chain", "draw"]) + base_level

# Plot the 94% HDI of the Hidden Physics
ax.fill_between(
    range(T),
    latent_hdi[:, 0], # Lower bound
    latent_hdi[:, 1], # Upper bound
    color="blue", alpha=0.3, label="Latent Physics 94% HDI"
)

# Plot the mean Latent Flow
ax.plot(latent_mean, color="blue", label="Latent Physics Mean")

ax.set_title("Hidden State vs Flawed Sensor (Real Units)")
ax.legend()
plt.show()

In [ ]:

loo_result = az.loo(trace)
khats = loo_result.pareto_k.values

# Identify structural failures where k > 0.7
high_k_indices = np.where(khats > 0.7)[0]
extreme_k_indices = np.where(khats > 1.0)[0]

print(f"[!] STRUCTURAL WARNING: {len(high_k_indices)} points exceed k=0.7 threshold.")
if len(extreme_k_indices) > 0:
    print(f"[X] CRITICAL LIABILITY: {len(extreme_k_indices)} points exceed k=1.0. Posterior is compromised.")
    for idx in extreme_k_indices:
        print(f" -> Index: {idx} | k-value: {khats[idx]:.2f} | Rain: {rainfall[idx]:.2f} | Flow: {flow_true[idx]:.2f}")
